In [5]:
import csv
from datetime import datetime

def convert_sevis_csv(input_file):
    """Convert SEVIS CSV file dates from DD/MM/YYYY to YYYY-MM-DD format"""
    
    # Read the original file
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames
    
    # Convert date fields
    date_fields = [
        'status_change_date',
        'date_of_birth', 
        'date_of_last_event',
        'program_start_date',
        'program_end_date',
        'initial_session_start_date',
        'current_session_start_date',
        'i_901_transaction_date'
    ]
    
    for row in rows:
        for field in date_fields:
            if row.get(field):
                date_str = row[field].strip()
                try:
                    # Convert from DD/MM/YYYY to YYYY-MM-DD
                    date_obj = datetime.strptime(date_str, '%d/%m/%Y')
                    row[field] = date_obj.strftime('%Y-%m-%d')
                except ValueError:
                    # Leave as is if can't parse
                    pass
    
    # Write back to the same file
    with open(input_file, 'w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"File '{input_file}' has been converted successfully!")
    print(f"Dates converted from DD/MM/YYYY to YYYY-MM-DD format")

# Usage
convert_sevis_csv('SEVIS_clean.csv')

File 'SEVIS_clean.csv' has been converted successfully!
Dates converted from DD/MM/YYYY to YYYY-MM-DD format


In [6]:
import csv
from datetime import datetime

def debug_sevis_dates(input_file):
    """Debug the date format in SEVIS CSV file"""
    
    print("DEBUGGING DATE FORMATS:")
    print("=" * 50)
    
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        
        # Check first 5 rows
        for i, row in enumerate(reader):
            if i >= 5:  # Only show first 5 rows
                break
                
            date_value = row.get('status_change_date', '')
            print(f"\nRow {i+1}: '{date_value}'")
            
            # Try different date formats
            formats_to_try = [
                '%d/%m/%Y',    # DD/MM/YYYY
                '%m/%d/%Y',    # MM/DD/YYYY
                '%Y-%m-%d',    # YYYY-MM-DD
                '%d-%m-%Y',    # DD-MM-YYYY
                '%Y/%m/%d',    # YYYY/MM/DD
            ]
            
            for fmt in formats_to_try:
                try:
                    date_obj = datetime.strptime(date_value.strip(), fmt)
                    print(f"  ✓ Matches format: {fmt}")
                    print(f"    Example: {date_obj.strftime('%Y-%m-%d')}")
                except:
                    continue

debug_sevis_dates('SEVIS_clean.csv')

DEBUGGING DATE FORMATS:

Row 1: '2024-06-27'
  ✓ Matches format: %Y-%m-%d
    Example: 2024-06-27

Row 2: '2024-07-23'
  ✓ Matches format: %Y-%m-%d
    Example: 2024-07-23

Row 3: '2024-03-21'
  ✓ Matches format: %Y-%m-%d
    Example: 2024-03-21

Row 4: '2024-05-28'
  ✓ Matches format: %Y-%m-%d
    Example: 2024-05-28

Row 5: '2024-05-02'
  ✓ Matches format: %Y-%m-%d
    Example: 2024-05-02


In [11]:
import csv
from datetime import datetime

def clean_applicant_csv(input_file):
    """
    Clean applicant CSV file:
    1. Convert date_of_birth from DD/MM/YYYY to YYYY-MM-DD
    2. Handle other potential date/time columns
    """
    
    print("Starting CSV cleaning process...")
    
    # Read the file
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames
    
    # Fix date_of_birth column (DD/MM/YYYY to YYYY-MM-DD)
    fixed_count = 0
    error_count = 0
    
    for row in rows:
        # 1. Fix date_of_birth
        dob = row.get('date_of_birth', '').strip()
        if dob:
            try:
                # Convert from DD/MM/YYYY to YYYY-MM-DD
                date_obj = datetime.strptime(dob, '%d/%m/%Y')
                row['date_of_birth'] = date_obj.strftime('%Y-%m-%d')
                fixed_count += 1
            except ValueError:
                try:
                    # Check if already in correct format
                    datetime.strptime(dob, '%Y-%m-%d')
                except ValueError:
                    error_count += 1
                    print(f"Warning: Could not parse date_of_birth: {dob}")
        
        # 2. Fix scientific notation numbers
        for key in ['phone_number', 'recieved_at', 'created_at', 'modified_at']:
            if key in row and row[key]:
                val = row[key]
                if 'E+' in str(val):  # Scientific notation
                    try:
                        # Convert from scientific notation to regular number
                        num = float(val)
                        row[key] = str(int(num))  # Convert to integer string
                    except:
                        pass
    
    # Write cleaned data back to the same file
    with open(input_file, 'w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"\nCleaning complete!")
    print(f"Date_of_birth conversions: {fixed_count}")
    print(f"Date parsing errors: {error_count}")
    print(f"File '{input_file}' has been cleaned and is ready for import.")

# Usage
clean_applicant_csv('APPLICANT_clean.csv')

Starting CSV cleaning process...

Cleaning complete!
Date_of_birth conversions: 0
Date parsing errors: 0
File 'APPLICANT_clean.csv' has been cleaned and is ready for import.


In [12]:
import csv
from datetime import datetime

def clean_all_dates_applicant_csv(input_file):
    """
    Clean ALL date/time columns in applicant CSV
    """
    
    print("Cleaning all date/time columns...")
    
    with open(input_file, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames
    
    # Date columns that need conversion
    date_columns = {
        'date_of_birth': '%d/%m/%Y',  # DD/MM/YYYY to YYYY-MM-DD
    }
    
    # DateTime columns (if you want to parse them)
    datetime_columns = [
        'recieved_at_datetime',
        'created_at_datetime', 
        'modified_at_datetime'
    ]
    
    for row in rows:
        # Convert simple dates
        for col, fmt in date_columns.items():
            if col in row and row[col]:
                val = row[col].strip()
                try:
                    date_obj = datetime.strptime(val, fmt)
                    row[col] = date_obj.strftime('%Y-%m-%d')
                except:
                    pass
        
        # Handle datetime strings (optional - keep as text if not needed)
        for col in datetime_columns:
            if col in row and row[col]:
                val = row[col].strip()
                if val:
                    # Convert from DD/MM/YYYY HH:MM to ISO format
                    try:
                        dt_obj = datetime.strptime(val, '%d/%m/%Y %H:%M')
                        row[col] = dt_obj.isoformat()  # YYYY-MM-DDTHH:MM:SS
                    except:
                        pass
    
    # Write back
    with open(input_file, 'w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"File '{input_file}' cleaned successfully!")

# Usage
clean_all_dates_applicant_csv('APPLICANT_clean.csv')

Cleaning all date/time columns...
File 'APPLICANT_clean.csv' cleaned successfully!


In [13]:
import csv

# Set ALL sevis_id to empty in your applicant.csv
with open('APPLICANT_clean.csv', 'r') as f:
    rows = list(csv.DictReader(f))

for row in rows:
    row['sevis_id'] = ''  # Set to empty/NULL

# Save it
with open('APPLICANT_clean.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

In [15]:
import csv
from datetime import datetime

def clean_deposit_csv(input_file):
    """
    Clean deposit CSV: Convert dates from DD/MM/YYYY to YYYY-MM-DD
    """
    with open(input_file, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    for row in rows:
        # Convert received_at (DD/MM/YYYY HH:MM → YYYY-MM-DD HH:MM:SS)
        if row.get('received_at'):
            try:
                dt = datetime.strptime(row['received_at'], '%d/%m/%Y %H:%M')
                row['received_at'] = dt.strftime('%Y-%m-%d %H:%M:%S')
            except:
                pass
        
        # Convert created_at (DD/MM/YYYY HH:MM → YYYY-MM-DD HH:MM:SS)
        if row.get('created_at'):
            try:
                dt = datetime.strptime(row['created_at'], '%d/%m/%Y %H:%M')
                row['created_at'] = dt.strftime('%Y-%m-%d %H:%M:%S')
            except:
                pass
        
        # Convert received_date (DD/MM/YYYY → YYYY-MM-DD)
        if row.get('received_date'):
            try:
                date_obj = datetime.strptime(row['received_date'], '%d/%m/%Y')
                row['received_date'] = date_obj.strftime('%Y-%m-%d')
            except:
                pass
    
    # Save cleaned file
    with open(input_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"Cleaned {len(rows)} rows in {input_file}")

# Run it
clean_deposit_csv('CONNECT_clean.csv')

Cleaned 6875 rows in CONNECT_clean.csv


In [16]:
import csv

def remove_duplicate_reference_ids(input_file):
    """Remove duplicate reference_id rows, keeping the first occurrence"""
    
    with open(input_file, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    # Track seen reference_ids
    seen_ids = set()
    unique_rows = []
    duplicate_count = 0
    
    for row in rows:
        ref_id = row['reference_id']
        if ref_id not in seen_ids:
            seen_ids.add(ref_id)
            unique_rows.append(row)
        else:
            duplicate_count += 1
            print(f"Duplicate found: reference_id={ref_id}")
    
    # Save cleaned file
    with open(input_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(unique_rows)
    
    print(f"\nRemoved {duplicate_count} duplicate rows")
    print(f"Kept {len(unique_rows)} unique rows")
    print(f"File cleaned: {input_file}")

# Run it
remove_duplicate_reference_ids('CONNECT_clean.csv')


Removed 0 duplicate rows
Kept 6875 unique rows
File cleaned: CONNECT_clean.csv


In [18]:
import csv

def remove_duplicate_reference_ids(input_file):
    """Remove duplicate reference_id rows, keeping the first occurrence"""
    
    with open(input_file, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    # Track seen reference_ids
    seen_ids = set()
    unique_rows = []
    duplicate_count = 0
    
    for row in rows:
        ref_id = row['reference_id']
        if ref_id not in seen_ids:
            seen_ids.add(ref_id)
            unique_rows.append(row)
        else:
            duplicate_count += 1
            print(f"Duplicate found: reference_id={ref_id}")
    
    # Save cleaned file
    with open(input_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(unique_rows)
    
    print(f"\nRemoved {duplicate_count} duplicate rows")
    print(f"Kept {len(unique_rows)} unique rows")
    print(f"File cleaned: {input_file}")

# Run it
remove_duplicate_reference_ids('CONNECT_clean.csv')


Removed 0 duplicate rows
Kept 6875 unique rows
File cleaned: CONNECT_clean.csv


In [19]:
import csv
import pandas as pd

def fix_duplicates_simple(input_file):
    """Simple pandas approach to remove duplicates"""
    
    # Read CSV with pandas
    df = pd.read_csv(input_file)
    
    print(f"Before: {len(df)} rows")
    print(f"Duplicate reference_ids: {df['reference_id'].duplicated().sum()}")
    
    # Remove duplicates, keep first
    df_clean = df.drop_duplicates(subset=['reference_id'], keep='first')
    
    print(f"After: {len(df_clean)} rows")
    print(f"Removed: {len(df) - len(df_clean)} duplicates")
    
    # Save back
    df_clean.to_csv(input_file, index=False)
    print(f"File saved: {input_file}")

# Run it
fix_duplicates_simple('CONNECT_clean.csv')

Before: 6875 rows
Duplicate reference_ids: 0
After: 6875 rows
Removed: 0 duplicates
File saved: CONNECT_clean.csv


In [20]:
import csv

def remove_duplicates_fixed(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    seen_ids = set()
    unique_rows = []
    duplicate_count = 0
    
    for row in rows:
        # Clean the reference_id - remove spaces, newlines, etc.
        ref_id = str(row['reference_id']).strip()
        row['reference_id'] = ref_id  # Update with cleaned version
        
        if ref_id not in seen_ids:
            seen_ids.add(ref_id)
            unique_rows.append(row)
        else:
            duplicate_count += 1
            print(f"Duplicate: {ref_id}")
    
    # Save
    with open(input_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(unique_rows)
    
    print(f"Removed {duplicate_count} duplicates, kept {len(unique_rows)} rows")

remove_duplicates_fixed('CONNECT_clean.csv')

Removed 0 duplicates, kept 6875 rows


In [21]:
import csv
from datetime import datetime

def fix_deposit_csv_dates(input_file):
    """Convert all dates from DD/MM/YYYY to YYYY-MM-DD format"""
    
    with open(input_file, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    fixed_count = 0
    
    for row in rows:
        # Fix received_at (14/04/2025 19:55 -> 2025-04-14 19:55:00)
        if row.get('received_at'):
            try:
                dt = datetime.strptime(row['received_at'], '%d/%m/%Y %H:%M')
                row['received_at'] = dt.strftime('%Y-%m-%d %H:%M:%S')
                fixed_count += 1
            except:
                pass
        
        # Fix created_at (19/02/2025 21:20 -> 2025-02-19 21:20:00)
        if row.get('created_at'):
            try:
                dt = datetime.strptime(row['created_at'], '%d/%m/%Y %H:%M')
                row['created_at'] = dt.strftime('%Y-%m-%d %H:%M:%S')
                fixed_count += 1
            except:
                pass
        
        # Fix received_date (14/04/2025 -> 2025-04-14)
        if row.get('received_date'):
            try:
                date_obj = datetime.strptime(row['received_date'], '%d/%m/%Y')
                row['received_date'] = date_obj.strftime('%Y-%m-%d')
                fixed_count += 1
            except:
                pass
    
    # Save fixed CSV
    with open(input_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"Fixed {fixed_count} date fields in {input_file}")
    print("Dates converted from DD/MM/YYYY to YYYY-MM-DD format")

# Run it
fix_deposit_csv_dates('CONNECT_clean.csv')

Fixed 20625 date fields in CONNECT_clean.csv
Dates converted from DD/MM/YYYY to YYYY-MM-DD format


In [22]:
import csv
from datetime import datetime

def clean_deposit_csv_for_upload(deposit_csv, applicant_csv):
    """
    1. Fix dates from DD/MM/YYYY to YYYY-MM-DD
    2. Set invalid reference_ids to empty (will become NULL)
    """
    
    # Read valid reference_ids from applicant table
    valid_ref_ids = set()
    try:
        with open(applicant_csv, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                valid_ref_ids.add(row['reference_id'].strip())
        print(f"Found {len(valid_ref_ids)} valid reference_ids in applicant table")
    except:
        print("Could not read applicant.csv, will clean dates only")
        valid_ref_ids = set()
    
    # Read and clean deposit CSV
    with open(deposit_csv, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    fixed_dates = 0
    fixed_refs = 0
    
    for row in rows:
        # Fix dates
        date_fields = ['received_at', 'created_at', 'received_date']
        for field in date_fields:
            if row.get(field):
                try:
                    if ' ' in row[field]:  # Has time part
                        dt = datetime.strptime(row[field], '%d/%m/%Y %H:%M')
                        row[field] = dt.strftime('%Y-%m-%d %H:%M:%S')
                    else:  # Date only
                        dt = datetime.strptime(row[field], '%d/%m/%Y')
                        row[field] = dt.strftime('%Y-%m-%d')
                    fixed_dates += 1
                except:
                    pass
        
        # Fix reference_id (set invalid ones to empty)
        ref_id = row.get('reference_id', '').strip()
        if ref_id and valid_ref_ids and ref_id not in valid_ref_ids:
            row['reference_id'] = ''
            fixed_refs += 1
            print(f"Set invalid reference_id {ref_id} to empty")
    
    # Save cleaned file
    with open(deposit_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"\nCleaning complete:")
    print(f"- Fixed {fixed_dates} date fields")
    print(f"- Fixed {fixed_refs} invalid reference_ids")
    print(f"File '{deposit_csv}' is ready for upload")

# Run it
clean_deposit_csv_for_upload('CONNECT_clean.csv', 'APPLICANT_clean.csv')

Found 6871 valid reference_ids in applicant table
Set invalid reference_id 1457541 to empty
Set invalid reference_id 1445768 to empty
Set invalid reference_id 1463640 to empty
Set invalid reference_id 1442267 to empty
Set invalid reference_id 1300808 to empty
Set invalid reference_id 1408958 to empty
Set invalid reference_id 1449891 to empty
Set invalid reference_id 1449036 to empty
Set invalid reference_id 1456256 to empty
Set invalid reference_id 1376458 to empty
Set invalid reference_id 1328705 to empty
Set invalid reference_id 1449883 to empty
Set invalid reference_id 1428720 to empty
Set invalid reference_id 1433382 to empty
Set invalid reference_id 1452610 to empty
Set invalid reference_id 1386745 to empty
Set invalid reference_id 1335336 to empty
Set invalid reference_id 1408693 to empty
Set invalid reference_id 1461793 to empty
Set invalid reference_id 1445524 to empty
Set invalid reference_id 1422492 to empty
Set invalid reference_id 1451554 to empty

Cleaning complete:
- Fixe

In [23]:
import csv
from datetime import datetime

def clean_outreach_csv(input_file):
    """
    Clean outreach CSV file:
    1. Fix appointment_date from DD/MM/YYYY to YYYY-MM-DD
    2. Handle empty values properly
    """
    
    with open(input_file, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    fixed_dates = 0
    fixed_sevis = 0
    
    for row in rows:
        # Fix appointment_date (DD/MM/YYYY to YYYY-MM-DD)
        appt_date = row.get('appointment_date', '').strip()
        if appt_date:
            try:
                date_obj = datetime.strptime(appt_date, '%d/%m/%Y')
                row['appointment_date'] = date_obj.strftime('%Y-%m-%d')
                fixed_dates += 1
            except ValueError:
                # If already in correct format or empty, leave as is
                try:
                    # Check if already YYYY-MM-DD
                    datetime.strptime(appt_date, '%Y-%m-%d')
                except ValueError:
                    # Try other formats
                    try:
                        date_obj = datetime.strptime(appt_date, '%m/%d/%Y')
                        row['appointment_date'] = date_obj.strftime('%Y-%m-%d')
                        fixed_dates += 1
                    except:
                        # Leave as is if can't parse
                        pass
        
        # Clean sevis_id - remove whitespace
        sevis_id = row.get('sevis_id', '').strip()
        if sevis_id:
            row['sevis_id'] = sevis_id
        else:
            # If empty, set to empty string (will become NULL)
            row['sevis_id'] = ''
            fixed_sevis += 1
        
        # Clean emails - remove whitespace
        if row.get('person_slu_email_address'):
            row['person_slu_email_address'] = row['person_slu_email_address'].strip()
        if row.get('person_email'):
            row['person_email'] = row['person_email'].strip()
    
    # Save cleaned file
    with open(input_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"Cleaned {len(rows)} rows")
    print(f"Fixed {fixed_dates} appointment dates")
    print(f"Found {fixed_sevis} empty sevis_ids")
    print(f"File '{input_file}' is ready for upload")

# Run it
clean_outreach_csv('OUTREACH_clean.csv')

Cleaned 1565 rows
Fixed 287 appointment dates
Found 0 empty sevis_ids
File 'OUTREACH_clean.csv' is ready for upload


In [24]:
import csv
from datetime import datetime

def clean_outreach_csv(input_file):
    """Clean outreach CSV: fix dates and set sevis_id to empty"""
    
    with open(input_file, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    
    for row in rows:
        # Fix appointment_date
        if row.get('appointment_date'):
            try:
                dt = datetime.strptime(row['appointment_date'], '%d/%m/%Y')
                row['appointment_date'] = dt.strftime('%Y-%m-%d')
            except:
                pass
        
        # SET ALL sevis_id TO EMPTY (will become NULL)
        row['sevis_id'] = ''
        
        # applicant_reference_id will be empty (NULL)
        if 'applicant_reference_id' in row:
            row['applicant_reference_id'] = ''
    
    # Save cleaned file
    with open(input_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"Cleaned {len(rows)} rows")
    print("All sevis_id values set to empty (will be NULL)")

# Run it
clean_outreach_csv('OUTREACH_clean.csv')

Cleaned 1565 rows
All sevis_id values set to empty (will be NULL)
